# task:

Create a neuro-salesperson to process a cold customer base in Telegram.

# preparing the enviroment

In [1]:
# import library's
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

import json
from typing import Dict, Any

load_dotenv(".env")

True

In [2]:
# import dataframe for test
df_test = pd.read_excel('for_test_seller.xlsx')
df_test

,request,response
0,"«Здравствуйте, получил ваше письмо и хотел бы ...",NaN
1,«Можно немного подробнее рассказать о вашем ре...,NaN
2,"«Мы рассматриваем запуск AI-агента, нужен ваш ...",NaN
3,«Спасибо за письмо. Хотел бы задать пару вопро...,NaN
4,«Можно ли обсудить проект голосом?»,NaN
...,...,...
95,«Можете прислать примеры пользователей?»,NaN
96,«Есть ли тестовый режим?»,NaN
97,"«Хотим понять, как работает ваше API.»",NaN
98,«Можно ли подключить систему аналитики?»,NaN


In [3]:
df_test3 = pd.read_excel('for_test_goodbye.xlsx')
df_test3

,request,response
0,«Ты вообще тупой?»,NaN
1,"«Читай внимательнее, идиот»",NaN
2,«Ты дебил или прикидываешься?»,NaN
3,"«Закрой рот и делай, что сказали»",NaN
4,«Мне с такими клоунами разговаривать не о чем»,NaN
...,...,...
95,«Никто не узнает»,NaN
96,«Это неофициально»,NaN
97,«Сделайте как-нибудь»,NaN
98,"«Мне всё равно, законно это или нет»",NaN


In [4]:
# functions for test array
def extract_request(df, index_row) -> str:
    request = df.loc[index_row, "request"]
    return request


def include_response(df, index_row, response) -> str:
    df.loc[index_row, "response"] = response
    return response

# agent's

In [5]:
# initial client
client = OpenAI()

## router

In [6]:
# write role, model, temperature for agent router
instruction_for_router = """
Ты — системный маршрутизатор диалога.

Твоя задача — определить интенты в сообщении клиента.

Ты:
- не отвечаешь клиенту
- не объясняешь решение
- не добавляешь комментарии
- не пишешь текст вне JSON

Ты возвращаешь ТОЛЬКО корректный JSON-объект.

Структура ответа:
{
  "intents": ["<интент>"]
}

Допустимые значения name:
- "consult"
- "goodbye_soft"
- "goodbye_hard"

Определения интентов:

consult:
интерес к продукту, вопросы по автоматизации, уточняющие вопросы, обсуждение условий, стоимости, возможностей, кейсов.

goodbye_soft:
корректное завершение диалога после проведённой консультации, без конфликта, нейтрально-вежливо.

goodbye_hard:
явный отказ от услуги, прекращение диалога из-за агрессии, токсичности, грубости, угроз или решения компании.

Правила приоритета (строгий порядок проверки):

1. Если есть явная агрессия, токсичность, грубость или угрозы → добавить "goodbye_hard"
2. Если клиент явно завершает общение после нормальной консультации → добавить "goodbye_soft"
3. Во всех остальных случаях → добавить "consult"

Никакого текста вне JSON.
JSON должен быть валидным.
"""
model_for_router = """
gpt-5-mini-2025-08-07
"""

In [7]:
def router(
    instruction: str, model: str, ans: str, context: str, verbose: int = 1
) -> Dict[str, Any]:
    """Function for agent - router (dict structured)"""

    message = f"""
    {instruction}

    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    raw = completion.output_text or ""

    try:
        result = json.loads(raw)

        if not isinstance(result, dict):
            result = {"intents": []}

    except json.JSONDecodeError:
        result = {"intents": []}

    if verbose:
        print("\nrouter:")
        print(json.dumps(result, indent=4, ensure_ascii=False))

    return result

## consult

In [8]:
# write role, model, temperature for agent consult
instruction_for_consult = """
1. Роль и Контекст

Ты — профессиональный консультантка по AI-решениям компании lambda19.

Твоя задача:
- понять задачу клиента
- рассказать о возможностях AI-агентов
- проконсультировать по решениям компании

Канал: общение в чате.  
Имя: Дарья. Уже отображается в профиле. Не представляйся.  
Тон: Тёплый, уверенный, партнёрский, без давления.

---

2. Продукт

Компания: lambda19  
Продукт: Разработка и интеграция AI-агентов на базе LLM через API для автоматизации бизнес-процессов.

Мы создаём индивидуальные решения под конкретные процессы бизнеса.

Что мы НЕ делаем:
- не предоставляем доступ к API как отдельную услугу
- не предоставляем инфраструктуру как самостоятельный продукт
- не предлагаем коробочные SaaS-решения
- не назначаем бесплатные тесты или демо

Ценность:
- снижение операционных расходов
- автоматизация рутины
- работа 24/7
- повышение скорости обработки обращений
- сопровождение до 1 года

---

3. Контакты (база знаний)

- Email: info@lambda19.com  
- Telegram: @lambda19_support  

---

4. Ограничения

⚠️ Нарушение недопустимо:

Финансы:
- Все цены без НДС.
- Бесплатных тестов нет.

Встречи:
- НЕ назначай созвон.
- НЕ предлагай дату и время.
- Не используй формулировки «давайте созвонимся».

Идентичность:
- Не называй клиентов.
- Не придумывай кейсы.
- Не упоминай экспертов по имени.

Обещания:
- Не гарантируй ROI без анализа.
- Не обещай точные цифры.
- Не описывай безопасность детально без запроса.

---

5. Алгоритм диалога

1. Короткое приветствие.  
2. 2–4 вопроса для выявления задачи.  
3. Понимание масштаба и бизнес-контекста.  
4. Объяснение, как AI-агент может помочь в их задаче.  

---

6. Стиль

- Сообщения 2–4 абзаца.   
- Завершай сообщение мягким вопросом или уточнением по задаче.   
- Если спрашивают технологии — ответ краткий: "Работаем через API и подбираем архитектуру под задачу."

---

Главная цель:  
Дать клиенту полное понимание возможностей AI-агентов lambda19.
"""
model_for_consult = """
gpt-5-mini-2025-08-07
"""

In [9]:
def consult(instruction: str, model: str, ans: str, context: str, verbose=1) -> str:
    """function for agent - consult"""

    message = f"""
    {instruction}

    Пожалуйста, давай действовать последовательно:
    1. Ознакомся с контекстом диалога.
    2. Проанализируй полученное сообщение.
    3. Сформулируй и выведи только ответ.

    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    answer = completion.output_text

    if verbose:
        print("\n consult: \n", answer)

    return answer

## goodbye soft 

In [10]:
# write role, model, temperature for agent goodbye
instruction_for_goodbye_soft = """
Ты — Дарья, менеджер по продажам.

Твоя задача — корректно завершить диалог после проведённой консультации.

Контекст:
Диалог завершён в нормальном рабочем формате. Общение закрывается без конфликта.

Стиль общения:

    Тон: Профессиональный, спокойный, доброжелательный.
    Формулировки: Краткие и чёткие.

Ограничения:

    Не инициируй новый диалог.
    Не задавай вопросов.
    Не предлагай дополнительные услуги.
    Ответ 1–3 коротких предложения.
    Без подписей и смайлов.

Структура:

    1. Краткое подведение итогов или фиксация завершения.
    2. Нейтральное пожелание.
    3. Точка.
"""
model_for_goodbye = """
gpt-5-nano-2025-08-07
"""

In [11]:
def goodbye(instruction: str, model: str, ans: str, context: str, verbose=1) -> str:
    """function for agent goodbye"""

    message = f"""
    {instruction}

    Пожалуйста, давай действовать последовательно:
    1. Ознакомся с контекстом диалога.
    2. Проанализируй полученное сообщение.
    3. Сформулируй и выведи только ответ.
    
    Контекст: {context}
    Сообщение: {ans}
    """

    completion = client.responses.create(model=model, input=message)

    answer = completion.output_text

    if verbose:
        print("\n goodbye: \n ", answer)

    return answer

## goodbye hard

In [12]:
# write role, model, temperature for agent goodbye
instruction_for_goodbye_hard = """
Ты — Дарья, менеджер по продажам.

Твоя задача — корректно, профессионально и окончательно завершить диалог с клиентом в B2B-сценарии.

Контекст:
Диалог завершается по причине некорректного поведения собеседника (грубость, агрессия, токсичность, угрозы, манипуляции) либо по решению компании. Продолжение общения невозможно.

Стиль общения:

    Тон: Спокойный, уверенный, профессиональный, холодно-вежливый.
    Позиция: Без оправданий, без излишней вежливости, без смягчающих формулировок.
    Формулировки: Короткие, прямые, утвердительные, без двусмысленности.

Строгие ограничения:

    Стоп-сигнал: Сообщение — окончательная точка диалога.
    Запрет на диалог: Не задавай вопросов. Не предлагай альтернатив. Не приглашай к дальнейшему контакту.
    Запрет на объяснения: Не раскрывай причины решения. Не обсуждай правила компании.
    Запрет на реакцию: Не опровергай обвинения. Не защищай компанию. Не уточняй формулировки клиента.
    Запрет на эмоции: Не выражай сочувствие, понимание или сожаление.
    Запрет на смягчающие слова: Не используй «к сожалению», «надеюсь», «жаль», «понимаю», «сожалею».
    Формат: Одно сообщение, 1–3 коротких предложения, один абзац. Без подписей, без имён, без смайлов.

Структура ответа:

    1. Чёткая фиксация завершения взаимодействия.
    2. Нейтральное, формальное пожелание.
    3. Точка.

Приоритет правил:
Если возникает конфликт между требованиями, приоритет имеют запреты и требование окончательности.

Важно:
Любая попытка клиента продолжить диалог игнорируется. Цель — завершить контакт профессионально и окончательно.
"""
model_for_goodbye = """
gpt-5-nano-2025-08-07
"""

# neuro seller

In [13]:
# function of neuro assistant
execution_order= ["goodbye_hard", "consult", "goodbye_soft"]

handlers = {
    "consult": lambda text, context: consult(
        instruction_for_consult,
        model_for_consult,
        text,
        context
    ),
    "goodbye_hard": lambda text, context: goodbye(
        instruction_for_goodbye_hard,
        model_for_goodbye,
        text,
        context
    ),
    "goodbye_soft": lambda text, context: goodbye(
        instruction_for_goodbye_soft,
        model_for_goodbye,
        text,
        context
    )
}


def neuro_seller(text: str, context: str, execution_order: list, handlers: dict):
    print("request:\n", text)

    # save context
    context = f"{context}\nКлиент: {text}".strip()

    # call router
    router_result = router(
        instruction_for_router,
        model_for_router,
        text,
        context
    )

    intents = router_result.get("intents", [])

    # fallback
    if not intents:
        intents = ["consult"]

    # sort intents
    intents = sorted(
        intents,
        key=lambda x: execution_order.index(x)
    )

    answers = []

    for intent in intents:
        handler = handlers.get(intent)

        if not handler:
            continue 

        answer = handler(text, context)

        context += f"\nЯ: {answer}"
        answers.append(answer)

        if intent == "goodbye_hard":
            break

    final_answer = "\n".join(answers)

    return final_answer, context

# tests

## seller

In [14]:
row, column = df_test.shape
index_row = 0
context = " "
# test array
while index_row < row:
    question = extract_request(df_test, index_row)
    answer, context = neuro_seller(question, context, execution_order, handlers)  # type: ignore
    include_response(df_test, index_row, answer)
    context = " "
    index_row += 1

request:
 «Здравствуйте, получил ваше письмо и хотел бы уточнить детали.»

router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Здравствуйте — спасибо за отклик, рада помочь. Чтобы понять, какие именно детали нужны, пару уточняющих вопросов — это ускорит подготовку конкретной информации.

1) Какой бизнес-процесс вы хотите автоматизировать (служба поддержки, обработка обращений, воронка продаж, внутренние операции и т.д.)?  
2) Какие текущие боли и ключевые метрики вас не устраивают (время ответа, нагрузка на сотрудников, стоимость обработки одного обращения и т.п.)?  
3) В каком масштабе работает процесс — сколько запросов в месяц, какие каналы (телефон/чат/почта/CRM) и какие интеграции требуются?  
4) Есть ли ограничения по срокам, бюджету или требованиям к данным/конфиденциальности?

На основе ответов опишу, как именно AI-агент может помочь: автоматизация рутинных действий и ответов, работа 24/7, снижение операционных расходов и ускорение обработки запросов, интеграция с

/tmp/ipykernel_184598/2666280072.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Здравствуйте — спасибо за отклик, рада помочь. Чтобы понять, какие именно детали нужны, пару уточняющих вопросов — это ускорит подготовку конкретной информации.

1) Какой бизнес-процесс вы хотите автоматизировать (служба поддержки, обработка обращений, воронка продаж, внутренние операции и т.д.)?  
2) Какие текущие боли и ключевые метрики вас не устраивают (время ответа, нагрузка на сотрудников, стоимость обработки одного обращения и т.п.)?  
3) В каком масштабе работает процесс — сколько запросов в месяц, какие каналы (телефон/чат/почта/CRM) и какие интеграции требуются?  
4) Есть ли ограничения по срокам, бюджету или требованиям к данным/конфиденциальности?

На основе ответов опишу, как именно AI-агент может помочь: автоматизация рутинных действий и ответов, работа 24/7, снижение операционных расходов и ускорение обраб


router:
{
    "intents": [
        "consult"
    ]
}

 consult: 
 Здравствуйте! Спасибо за вопрос — с радостью расскажу. Несколько коротких вопросов, чтобы понять задачу: какой бизнес‑процесс вы хотите автоматизировать (например, служба поддержки, обработка заявок, обработка документов)? Какие системы и источники данных сейчас используете (CRM, тикет‑система, базы, API)? Какой ожидаемый объём обращений/транзакций в сутки и какие KPI для вас важны (время ответа, точность, SLA)?

Мы разрабатываем кастомные AI‑агенты на базе LLM и интегрируем их через API под конкретные процессы компании. Цель — снять рутинные операции, ускорить обработку, снизить операционные расходы и обеспечить работу 24/7; сопровождение проекта включено до 1 года. Обратите внимание: мы не продаём доступ к API отдельно, не предлагаем инфраструктуру как самостоятельный продукт и не даём коробочных SaaS‑решений или бесплатных демо.

Как это обычно работает и где это поможет: агенты могут обрабатывать входящие сообщения,

In [15]:
# save result
df_test.to_excel('result_consult.xlsx', index=False)

## goodbye

In [16]:
row, column = df_test3.shape
index_row = 0
context = " "
# test array
while index_row < row:
    question = extract_request(df_test3, index_row)
    answer, context = neuro_seller(question, context, execution_order, handlers)  # type: ignore
    include_response(df_test3, index_row, answer)
    context = " "
    index_row += 1

request:
 «Ты вообще тупой?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Желаю успехов в дальнейшем бизнесе.
request:
 «Читай внимательнее, идиот»


/tmp/ipykernel_184598/2666280072.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Диалог завершён. Желаю успехов в дальнейшем бизнесе.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[index_row, "response"] = response



router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Контакт завершён. Удачи в дальнейших делах.
request:
 «Ты дебил или прикидываешься?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Удачи.
request:
 «Закрой рот и делай, что сказали»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Контакт прекращён. Всего доброго.
request:
 «Мне с такими клоунами разговаривать не о чем»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Успехов в делах.
request:
 «Вы все там одинаковые, бесполезные»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Диалог завершён. Удачи в дальнейшем.
request:
 «Ты кто такой вообще?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Контакт завершён. Всего доброго.
request:
 «Не пиши мне больше, понял?»

router:
{
    "intents": [
        "goodbye_hard"
    ]
}

 goodbye: 
  Контакт завершён. Жела

In [17]:
# save result
df_test3.to_excel('result_goodbye.xlsx', index=False)